# Programming Python for Bioinformatics (WBT-MBT2-25E)

---

## Python Programming Crash Course
### Wojciech Dec
<br>

### Part 4 - Exceptions, Error Handling, and Basic File Operations

We will become familiar with the keywords:
* `try`, `with`, `assert`, `except`, `raise`, `finally`

built-in functions:
* `open`

and standard library modules:
* `pathlib`, `datetime`


### Errors and Exceptions

In Python, these terms are often used interchangeably. You have likely encountered many already. Exceptions are errors detected during program execution. They are objects derived from `BaseException`, which allows defining custom exception classes.

In Python, exception classes form a hierarchy; see below for details:
https://docs.python.org/3/library/exceptions.html

Exceptions can be caught (anticipated and handled). This is done using the **try–except–else–finally** construct, analogous to if–elif–else, but using exception classes instead of conditions.

In [1]:
try:
    # code that may raise an exception
    pass

except ExpectedException:
    # handle a specific exception
    pass

except AnotherExpectedException as err:
    # handle another exception (err stores the exception object)
    pass

else:
    # executed if no exception occurred
    pass

finally:
    # always executed
    pass

In [3]:
def my_divide_function(a, b):
    try:
        return a / b

    except ZeroDivisionError:
        print("Division by zero is not allowed.")
        return

    except TypeError:
        print("Division is undefined for the given types.")
        return

my_divide_function(1, "0")
my_divide_function(1, 0)

Division is undefined for the given types.
Division by zero is not allowed.


In some cases, we may want to explicitly raise an exception; the `raise` statement is used for this purpose.

In [4]:
def better_divide_function(a, b):
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        raise TypeError("Inputs must be numeric.")
    if b == 0:
        raise ZeroDivisionError("Division by zero.") # in general raise ExceptionClass("message")

    return a / b

In some cases, we may want an exception to terminate the program, but only after performing cleanup (e.g., saving data). The exception can then be re-raised.

In [ ]:
# Catch an exception to perform cleanup, then re-raise it using `raise`

try:
    do_something_dangerous()

except Exception:
    do_something_to_apologize()
    raise  # re-raise the same exception

**Avoid bare `except`**
* replace with `except Exception` - this prevents catching system-level exceptions (e.g., `KeyboardInterrupt`)

In [ ]:
# `finally` is prefered for guaranteed cleanup
try:
    do_something_dangerous()
finally:
    save_results()

We can use the `assert` statement to raise an exception when a condition is not met.

In [8]:
x = "unfortunately"

assert x != "unfortunately", "Condition not satisfied."

AssertionError: Condition not satisfied.

Here, the condition is false, so an `AssertionError` is raised with the given message.

#### Best practices
* Reserve `assert` for internal checks and debugging
* When appropriate, raise standard exceptions such as `ValueError`, `TypeError`, or `FileNotFoundError` instead of generic ones
* Use `finally` for cleanup that must happen regardless of success or failure
* Prefer `raise` when invalid input or state should stop execution
* Catch specific exceptions, not `except`

### Basic File Operations

To open or create files, we use the built-in function `open`. Its key arguments are the **file path** and a **mode string**.

|||
|--|:--|
|'r'    |   open for reading (default)|
|'w'     |  open for writing, truncating the file first|
|'x'     |  create a new file and open it for writing|
|'a'     |  open for writing, appending to the end of the file if it exists|
|'b'     |  binary mode|
|'t'     |  text mode (default)|
|'+'     |  open a disk file for updating (reading and writing)|

**Note**: Files should always be closed after use. Selecting an incorrect mode may overwrite data.

In [ ]:
fp = open('diary.txt', 'x')
fp.write("Dear diary")
fp.close()

fp = open('diary.txt', 'r')
data = fp.read()
fp.close()
print(data)

### Context manager (`with`)

The `with` statement simplifies file handling (context manager):

In [ ]:
# in general

with open(path, mode) as name:
    # operations on file
    pass

In [ ]:
with open('filename.txt') as file:  # default: mode='r'
    print(file.read())

This is equivalent to:

In [ ]:
try:
    file = open('filename.txt')
    file.read()
finally:
    file.close()

#### Reading lines

Methods `readlines()` and `readline()` provide more control over input. Use `for` for simplicity (most memory-efficient), `readline()` for control, and avoid `readlines()` for large files.

In [2]:
data = """>seq1 Homo sapiens gene
ATGCGTAGCTAG
ATGCGTAGATGCGTAGCTATTT
>seq2 Mus musculus gene
ATTTGGGCCA
>seq3 synthetic
NNNNATGC
"""

with open('example.fasta', 'x') as f:
    f.write(data)

with open('example.fasta') as f:
    sequences = {}
    current_id = None

    for line in f:
        line = line.strip()
        if line.startswith(">"):
            current_id = line[1:]
            sequences[current_id] = ""
        else:
            sequences[current_id] += line

print(sequences)

{'seq1 Homo sapiens gene': 'ATGCGTAGCTAGATGCGTAGATGCGTAGCTATTT', 'seq2 Mus musculus gene': 'ATTTGGGCCA', 'seq3 synthetic': 'NNNNATGC'}


### Working with system paths (`pathlib`)

`pathlib` provides an object-oriented interface for filesystem paths. `Path` objects replace string-based paths.
The `/` operator is overloaded - constructed paths are automatically adapted to the operating system.

In [ ]:
from pathlib import Path

path = Path.cwd() / "folder" # cwd = current working directory
path.resolve()  # convert relative path to absolute

print(path, path.exists())
path.mkdir(exist_ok=True)
print(path.is_dir())

path = path / "file.txt"
print(path.is_file())
path.touch()
print(path, path.exists())

path.write_text("example")
print(path.read_text())

print(path.suffix, path.parts)

Iterating over directory contents

In [ ]:
path = Path.cwd() 

for child in path.iterdir():
    if child.is_dir():
        print(child.parts)  # returned objects are Path instances
        
    elif child.is_file():
        print(child.name, child.stat().st_size)

#### `pathlib.glob` — pattern-based file search

In [ ]:
path = Path.cwd()
files = list(path.glob("*.ipynb"))
print(files)

| Pattern   | Meaning                     |
| --------- | --------------------------- |
| `*`       | any characters              |
| `?`       | single character            |
| `*.txt`   | all `.txt` files            |
| `data_*`  | files starting with `data_` |
| `*.fasta` | all FASTA files             |


#### Recursive search

In [ ]:
files = list(path.glob("**/*.txt")) # ** → search recursively in all subdirectories
# or
path.rglob("*.txt")  # rglob → equivalent to glob("**/*.txt")

#### Moving files
To move files, one can use `shutil`; however, `pathlib` also provides this functionality. `shutil` is better for cross-filesystem operations.

In [ ]:
import shutil

shutil.move("file.txt", "new_dir/file.txt")

In [ ]:
Path("file.txt").replace("new_dir/file.txt")

#### `zipfile`
The `zipfile` module is used to create and extract ZIP archives.

In [8]:
from zipfile import ZipFile

path = Path.cwd() / "pp4b_4_files.zip"

with ZipFile(path, "r") as z:
    z.extractall("extracted/")

### Date and Time (`datetime`)

We can manipulate dates and times using standard library modules.

In [17]:
import time
from datetime import date, datetime

today = date.today()
print(today)

print(today == date.fromtimestamp(time.time()))  # reminder: time.time() → seconds since epoch

my_birthday = date(today.year, 8, 22)

if my_birthday < today:
    my_birthday = my_birthday.replace(year=today.year + 1)

print(my_birthday)

time_to_birthday = my_birthday - today
print(f"Birthday in {time_to_birthday.days} days.")

2026-03-30
True
2026-08-22
Birthday in 145 days.


In [10]:
with open("diary.txt", "a") as fp:  # append mode
    fp.write(str(date.today()))
    fp.write(" I ate a sandwich\n")

In [18]:
path = Path("diary.txt")

last_modified = datetime.fromtimestamp(path.stat().st_mtime)
print(last_modified)

2026-03-30 00:23:59.808761


**Exercise 0.** Use the `pathlib` module and `zipfile` to extract files from `pp4b_4_files.zip`.

**Exercise 1.** Write a function that reads the `scop.txt` file and counts the number of occurrences of each fold class (the label column). The function should accept a `pathlib.Path` object and return a dictionary `{label: count}`.

**Exercise 2.** Write a function that reads a FASTA file (`data.fasta`) and returns a dictionary `{accession: sequence_length}`. The function should accept a `pathlib.Path` object as input. If the file does not exist, raise a `FileNotFoundError`.

**Exercise 3.** Write a function that reads a CSV-like file (`stability.txt`) containing protein sequences and stability values and returns a list of `(sequence, label)` tuples. The function should accept a `pathlib.Path` object. Invalid rows should be skipped, not terminate the program.

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

**Exercise 4.** Write a function that appends a new diary entry to a file. The function should take a string (`entry`) and a `pathlib.Path object`. Each entry should be written on a new line, prefixed with the current date and time. If the file does not exist, it should be created.

Hints:
* Append mode (`'a'`)
* Add timestamp (`datetime.now()`)
* Each entry in format:
[YYYY-MM-DD HH:MM:SS] - entry text